In [1]:
import os
os.chdir("/Users/jakubdurczok/Documents/GitHub/songs/legacy")

from main import check_ffmpeg, extract_apple_playlist, search_youtube, download_youtube_audio

import requests
import argparse
import pyfiglet
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed
import json
import sys
import re
import yt_dlp
from tqdm import tqdm
import logging
import os
import shutil

In [2]:
import logging
import requests
from bs4 import BeautifulSoup
import json
logging.info("Fetching playlist data...")
playlist_url = "https://music.apple.com/pl/playlist/favourite-107950/pl.u-xlyNqGYue3GNz0"
response = requests.get(playlist_url)
soup = BeautifulSoup(response.text, "html.parser")

script_tag = soup.find("script", id="serialized-server-data")

if script_tag:
    # get the JSON string
    json_str = script_tag.get_text()
else:
    logging.error(
        "No script tag found in the HTML. Ensure Valid/Public Playlist URL"
    )
    raise ValueError(
        "No script tag found in the HTML. Ensure Valid/Public Playlist URL"
    )

if json_str:
    # convert the JSON string into a Python dictionary
    data = json.loads(json_str)

Fetching playlist data...


In [3]:
from main import _get_apple_music_bearer_token, _fetch_remaining_playlist_tracks

# Apple Music only server-renders the first ~300 tracks in this section.
page_data = data["data"][0]["data"]
playlistJson = page_data["sections"][1]["items"]

# Fetch the remaining tracks (if any) via the catalog API.
next_intent = page_data.get("nextIntent")
if next_intent and next_intent.get("$kind") == "PlaylistPaginationIntent":
    bearer_token = _get_apple_music_bearer_token(playlist_url)
    if bearer_token:
        more_songs, more_artists = _fetch_remaining_playlist_tracks(next_intent, bearer_token)
        playlistJson += [
            {"title": title, "artistName": artist}
            for title, artist in zip(more_songs, more_artists)
        ]

In [4]:
data

{'data': [{'intent': {'$kind': 'PlaylistDetailPageIntent',
    'contentDescriptor': {'kind': 'playlist',
     'identifiers': {'storeAdamID': 'pl.u-xlyNqGYue3GNz0'},
     'locale': {'storefront': 'pl', 'language': None}},
    'prominentItemIdentifier': None},
   'data': {'pageMetrics': {'instructions': [{'data': {'topic': 'xp_its_music_main',
        'shouldFlush': False,
        'fields': {'eventType': 'page'},
        'includingFields': ['pageFields', 'languages'],
        'excludingFields': []},
       'invocationPoints': ['pageEnter', 'appEnter']},
      {'data': {'topic': 'xp_its_music_main',
        'shouldFlush': False,
        'fields': {'eventType': 'impressions'},
        'includingFields': ['pageFields', 'languages', 'impressions'],
        'excludingFields': []},
       'invocationPoints': ['pageExit', 'appExit']}],
     'pageFields': {'pageType': 'Playlist',
      'pageUrl': 'https://music.apple.com/pl/playlist/favourite-107950/pl.u-xlyNqGYue3GNz0',
      'pageFeatureName':

In [5]:
len(playlistJson)

364

In [5]:
os.listdir("/Volumes/SWIM PRO")

['System Volume Information',
 '.fseventsd',
 '.Spotlight-V100',
 'Céline Dion - Because You Loved Me (Theme from ＂Up Close and Personal＂)(Audio).mp3',
 '._Céline Dion - Because You Loved Me (Theme from ＂Up Close and Personal＂)(Audio).mp3',
 "Céline Dion - I'm Alive (Official Audio).mp3",
 "._Céline Dion - I'm Alive (Official Audio).mp3",
 "Céline Dion - That's The Way It Is (Official HD Video).mp3",
 "._Céline Dion - That's The Way It Is (Official HD Video).mp3",
 'Peter Schilling - Major Tom (Völlig losgelöst...) ｜\xa01st German Video Version 1982.mp3',
 '._Peter Schilling - Major Tom (Völlig losgelöst...) ｜\xa01st German Video Version 1982.mp3',
 'Phil Collins - Against All Odds (Take A Look At Me Now) (Official Music Video).mp3',
 '._Phil Collins - Against All Odds (Take A Look At Me Now) (Official Music Video).mp3',
 'Phil Collins - Another Day In Paradise (2016 Remaster Turquoise Vinyl Edition).mp3',
 '._Phil Collins - Another Day In Paradise (2016 Remaster Turquoise Vi

In [2]:
url = "https://music.apple.com/pl/playlist/favourite-107950/pl.u-xlyNqGYue3GNz0"
max_threads = 8
output_dir = "/Users/jakubdurczok/Documents/GitHub/songs/data"

logging.info(f"Extracting Music Using {max_threads} threads")
logging.info(f"Output Directory: {output_dir}\n")

check_ffmpeg()
logging.info("Extracting playlist songs and artists...")
songs, artists = extract_apple_playlist(url)

# Parallel YouTube search
logging.info("Searching YouTube for song URLs...")
yt_urls = search_youtube(songs, artists, max_threads=max_threads)

# # Parallel audio download
# # os.makedirs(output_dir, exist_ok=True)
# # logging.info("Starting downloads...")
download_youtube_audio(yt_urls, output_dir=output_dir, max_threads=max_threads)

# logging.info("All downloads completed successfully!")

Extracting Music Using 8 threads
Output Directory: /Users/jakubdurczok/Documents/GitHub/songs/data

Extracting playlist songs and artists...
Fetching playlist data...
Found 300 songs in the playlist.
Searching YouTube for song URLs...


[{'id': 'track-lockup - pl.u-xlyNqGYue3GNz0 - 1446014714', 'title': '99 Luftballons', 'trackNumber': None, 'tertiaryLinks': [{'title': 'Nena', 'segue': {'$kind': 'flowAction', 'destination': {'kind': 'catalogItemDetailPage', 'contentDescriptor': {'kind': 'album', 'identifiers': {'storeAdamID': '1446014467'}, 'url': 'https://music.apple.com/pl/album/99-luftballons/1446014467', 'locale': {'storefront': 'pl', 'language': None}}, 'prominentItemIdentifier': None}, 'actionMetrics': {'data': [{'fields': {'actionUrl': 'https://music.apple.com/pl/album/99-luftballons/1446014467', 'actionType': 'navigate', 'eventType': 'click', 'targetType': 'link', 'targetId': '1446014467', 'eventVersion': 5, 'actionDetails': {'kind': 'album'}}, 'includingFields': ['pageFields', 'impressionsSnapshot', 'languages', 'clickLocation'], 'excludingFields': [], 'topic': 'xp_its_music_main', 'shouldFlush': False}], 'custom': {}}}}], 'duration': 231467, 'contentDescriptor': {'kind': 'song', 'identifiers': {'storeAdamID'

Downloading:   0%|          | 0/300 [00:00<?, ?it/s][youtube] No supported JavaScript runtime could be found. Only deno is enabled by default; to use another runtime add  --js-runtimes RUNTIME[:PATH]  to your command/config. YouTube extraction without a JS runtime has been deprecated, and some formats may be missing. See  https://github.com/yt-dlp/yt-dlp/wiki/EJS  for details on installing one
[youtube] No supported JavaScript runtime could be found. Only deno is enabled by default; to use another runtime add  --js-runtimes RUNTIME[:PATH]  to your command/config. YouTube extraction without a JS runtime has been deprecated, and some formats may be missing. See  https://github.com/yt-dlp/yt-dlp/wiki/EJS  for details on installing one
[youtube] No supported JavaScript runtime could be found. Only deno is enabled by default; to use another runtime add  --js-runtimes RUNTIME[:PATH]  to your command/config. YouTube extraction without a JS runtime has been deprecated, and some formats may be 

In [3]:
yt_urls

['https://www.youtube.com/watch?v=FT3D1Cu6g10',
 'https://www.youtube.com/watch?v=OcZFpkHEhLA',
 'https://www.youtube.com/watch?v=EbZP13EbpgM',
 'https://www.youtube.com/watch?v=f2dwSG1x984',
 'https://www.youtube.com/watch?v=TMSlTQmDLx4',
 'https://www.youtube.com/watch?v=E4OzdyxbOuU',
 'https://www.youtube.com/watch?v=RYk0_U7kf3g',
 'https://www.youtube.com/watch?v=l6Qp48VoJb4',
 'https://www.youtube.com/watch?v=Lgz455Iu8UY',
 'https://www.youtube.com/watch?v=TYQzIw0zat0',
 'https://www.youtube.com/watch?v=b9UcXl6lQgE',
 'https://www.youtube.com/watch?v=iBVGwHFT144',
 'https://www.youtube.com/watch?v=va3sgHayM7k',
 'https://www.youtube.com/watch?v=qBTJkfuC5Ds',
 'https://www.youtube.com/watch?v=FZMPtvJe0fg',
 'https://www.youtube.com/watch?v=hQLGCX8D-1Y',
 'https://www.youtube.com/watch?v=jRqhGC5vgC0',
 'https://www.youtube.com/watch?v=nwrqQ2jYpwY',
 'https://www.youtube.com/watch?v=G_UXvcr22rM',
 'https://www.youtube.com/watch?v=81WhM9dOcYI',
 'https://www.youtube.com/watch?v=UJLtEp